In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import warnings

from pathlib import Path

import pandas as pd
import scanpy as sc
import numpy as np
import zarr

from dotenv import dotenv_values
from matplotlib import pyplot as plt
from scipy.ndimage import gaussian_filter
from tqdm import tqdm

warnings.simplefilter(action='ignore', category=FutureWarning)
sc.set_figure_params(frameon=False, dpi=120, figsize=(5, 5))

In [4]:
config_path = Path("config.env")
if not config_path.exists():
    config_path = Path("../../../config.env")
CONFIG = dotenv_values(config_path)
assert CONFIG != {}, 'Configuration is empty - check the filepath.'

DATADIR = Path(CONFIG['DATADIR'])

OUTDIR = DATADIR / "7_spatial"

# Notebook constants
ROOT_REGION_ID = 997

SIGMA = 2
MIN_SUPPORT = 0.15

### Load objects

In [5]:
# Load voxel grid Zarr store
voxel_store = zarr.open(OUTDIR / 'sp_voxels.zarr', mode='r')
voxel_store.tree()

/
├── L1_mass (6, 168, 144, 107) float32
├── L1_prob (6, 168, 144, 107) float32
├── L1_rel_mass (6, 168, 144, 107) float32
├── L2_mass (82, 168, 144, 107) float32
├── L2_prob (82, 168, 144, 107) float32
├── L2_rel_mass (82, 168, 144, 107) float32
├── L3_mass (274, 168, 144, 107) float32
├── L3_prob (274, 168, 144, 107) float32
├── L3_rel_mass (274, 168, 144, 107) float32
├── L4_mass (716, 168, 144, 107) float32
├── L4_prob (716, 168, 144, 107) float32
├── L4_rel_mass (716, 168, 144, 107) float32
├── L5_mass (932, 168, 144, 107) float32
├── L5_prob (932, 168, 144, 107) float32
├── L5_rel_mass (932, 168, 144, 107) float32
├── counts (168, 144, 107) float32
├── evidence (168, 144, 107) float32
├── latent_embedding (45, 168, 144, 107) float32
├── region_labels (168, 144, 107) uint32
└── region_masks (176, 168, 144, 107) bool

In [6]:
# Extract root mask and derive voxel shape from it
idx = voxel_store['region_masks'].attrs['ids'].index(ROOT_REGION_ID) # root region (all region encompassing)
root_mask = voxel_store['region_masks'][idx, ...]

nx, ny, nz = root_mask.shape
n_voxels = nx * ny * nz

print((nx, ny, nz), n_voxels)

(168, 144, 107) 2588544


In [52]:
# Load concatenated spatial datasets
adata = sc.read_h5ad(OUTDIR / 'SP_concat.h5ad')
adata

AnnData object with n_obs × n_vars = 1401313 × 56939
    obs: 'Dataset', 'Sample', 'Author', 'Technology', 'Donor', 'Sex', 'RegionID', 'RegionAcronym', 'RegionName', 'total_counts', 'total_hvg_counts', 'total_genes', 'IsHY'
    var: 'gene_stable_ids', 'IsHVG'
    uns: 'pathmat'
    obsm: 'P_path', 'X_embed', 'X_umap', 'X_umap_sc', 'coords_ccf_mm', 'coords_ccf_mm_wrap', 'coords_source', 'coords_voxels_25um', 'coords_voxels_25um_bounded'
    varm: 'IsCaptured'

In [45]:
mask_langlieb = adata.obs['Dataset'] == 'SP2-04'
adata.obs.loc[
    mask_langlieb,
    'Sex'
] = 'F'

In [46]:
adata = adata[
    adata.obs['Sex'] == 'F'
]

### Sample-wise voxel aggregation

Full grid size in gigabytes can be computed by `nc * nx * ny * nz * 4 / 1024**3`. In our case: 7 gb.

In [47]:
# Intialize empty total grids
# total_mass = csr_matrix((n_paths, n_voxels), dtype=np.float32)
total_coverage = np.zeros(n_voxels, dtype=np.int32)

unique_sample = adata.obs['Sample'].dropna().unique()
for sample in tqdm(unique_sample):
    
    sample_indices = np.where(adata.obs['Sample'] == sample)[0]
    n_points = len(sample_indices)
    
    # Subset and extract only voxel coords for that sample
    subset = adata[sample_indices]
    coords = subset.obsm['coords_voxels_25um_bounded'].values
    ravel_coords = np.ravel_multi_index(coords.T, dims=(nx, ny, nz))
    
    # Construct counts grid
    sample_counts = np.bincount(
        ravel_coords,
        minlength=n_voxels
    ).astype(np.int32)
    
    # Infer coverage grid from counts grid
    sample_coverage = (sample_counts > 0).astype(np.int32)
    
    # Add intermediate grid to totals
    total_coverage += sample_coverage

100%|██████████| 84/84 [00:27<00:00,  3.10it/s]


### Setting up support matrix

This gets confusing - there are to grids:

1. Support this is the actual normalization matrix
2. Evidence this provides a reference of how well sampled a voxel was accounting for OOB regions

In [48]:
sampled_mask = (total_coverage > 0).astype(np.float32) # binary matrix 1 - point was sampled in this voxel 0 - missing
sampled_mask = sampled_mask.reshape(nx, ny, nz)

smoothed_support = gaussian_filter(sampled_mask, sigma=SIGMA, mode='constant', cval=0)
smoothed_root_mask = gaussian_filter(root_mask.astype(np.float32), sigma=SIGMA, mode='constant', cval=0)

# Set OOB to nan to propagate to all further grids
smoothed_support[~root_mask] = np.nan

# Normalized convolution - we don't care for voxels out of bound (not in brain/root)
smoothed_evidence = smoothed_support / (smoothed_root_mask + 1e-8)

In [49]:
idx = voxel_store['region_masks'].attrs['ids'].index(1097) # root region (all region encompassing)
hy_mask = voxel_store['region_masks'][idx, ...]

In [50]:
valid_grid = smoothed_evidence > MIN_SUPPORT
valid_hy_arr = valid_grid[hy_mask]

np.sum(valid_hy_arr), len(valid_hy_arr)

(np.int64(289894), 488718)